# Sparse SAE Feature Space EMI Pipeline

Two-phase pipeline: builds sparse prototype vectors in SAE feature space from W2V-selected exemplars, then scores a temporally stratified set of speeches.

**Phase 1** — select top 100K evidence + 100K intuition speeches by W2V EMI → collect BERT/GPT-2 activations → pass through SAE → average to get sparse prototype vectors  
**Phase 2** — sample 5K speeches/decade × 14 decades = 70K → collect activations → SAE features → cosine EMI in sparse feature space

In [ ]:
# Cell 1 — Imports
import os, sys, json, glob, gc, time, logging, random, csv
csv.field_size_limit(sys.maxsize)

from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import roc_auc_score

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s'
)
logger = logging.getLogger(__name__)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 2 — Configuration
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Phase 1
PHASE1_TOP_N_PER_POLE       = 100_000   # 200K total exemplars

# Phase 2
PHASE2_SPEECHES_PER_DECADE  = 5_000
PHASE2_DECADE_START         = 1880
PHASE2_DECADE_END           = 2020      # inclusive

# Batch sizes
BERT_BATCH_SIZE  = 64
GPT2_BATCH_SIZE  = 32
SAE_BATCH_SIZE   = 1024

# Input paths
DATA_CSV         = './data/raw_data/filtered_speeches.csv'
W2V_PATH         = './data/embeddings/congressional_embeddings.txt'
BERT_MODEL_DIR   = './outputs/bert/final'
GPT2_MODEL_DIR   = './outputs/gpt2/final'

# SAE paths — auto-detected below
SAE_BERT_DIR     = './activations/sae/bert'
SAE_GPT2_DIR     = './activations/sae/gpt2'
SAE_BERT_PATH    = None
SAE_GPT2_PATH    = None

# Output dir
RESULTS_DIR = './results/sparse_sae_emi'
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'Results will be written to: {RESULTS_DIR}')

In [ ]:
# Cell 3 — Path verification + SAE auto-detection
def _auto_detect_sae(sae_dir, model_tag):
    pattern = os.path.join(sae_dir, f'sae_{model_tag}_*.pt')
    files = sorted(glob.glob(pattern), key=os.path.getmtime)
    if not files:
        return None
    chosen = files[-1]  # most recently modified
    logger.info('Auto-detected %s SAE: %s', model_tag, chosen)
    return chosen

SAE_BERT_PATH = _auto_detect_sae(SAE_BERT_DIR, 'bert')
SAE_GPT2_PATH = _auto_detect_sae(SAE_GPT2_DIR, 'gpt2')

errors = []

for name, path in [
    ('Data CSV',      DATA_CSV),
    ('W2V embeddings', W2V_PATH),
    ('BERT config',   os.path.join(BERT_MODEL_DIR, 'config.json')),
    ('GPT-2 config',  os.path.join(GPT2_MODEL_DIR, 'config.json')),
]:
    if not os.path.isfile(path):
        errors.append(f'Missing {name}: {path}')
    else:
        size_mb = os.path.getsize(path) / 1e6
        print(f'  OK  {name:<20} ({size_mb:.1f} MB)  {path}')

for name, path in [('BERT SAE', SAE_BERT_PATH), ('GPT-2 SAE', SAE_GPT2_PATH)]:
    if path is None or not os.path.isfile(path):
        errors.append(f'Missing {name}: not found in {SAE_BERT_DIR if "BERT" in name else SAE_GPT2_DIR}')
    else:
        size_mb = os.path.getsize(path) / 1e6
        print(f'  OK  {name:<20} ({size_mb:.1f} MB)  {path}')

if errors:
    print('\nERRORS:')
    for e in errors:
        print(f'  {e}')
    raise FileNotFoundError(f'{len(errors)} required file(s) missing — fix paths and rerun')

print('\nAll required files verified.')

In [ ]:
# Cell 4 — Shared helper functions and class definitions

# ── SparseAutoencoder ─────────────────────────────────────────────────────────
# Must match the class in train_sae.ipynb exactly
class SparseAutoencoder(nn.Module):
    def __init__(self, d_in, d_sae):
        super().__init__()
        self.d_in  = d_in
        self.d_sae = d_sae
        self.W_enc = nn.Linear(d_in, d_sae, bias=True)
        self.W_dec = nn.Linear(d_sae, d_in, bias=True)

    def encode(self, x):
        return F.relu(self.W_enc(x))

    def forward(self, x):
        f = self.encode(x)
        return f, self.W_dec(f)


def load_sae_checkpoint(path):
    """Load SAE from checkpoint. Returns (sae, act_mean) where act_mean shape is (1, d_in)."""
    ckpt     = torch.load(path, map_location=device, weights_only=False)
    cfg      = ckpt['config']
    sae      = SparseAutoencoder(cfg['d_in'], cfg['d_sae']).to(device)
    sae.load_state_dict(ckpt['model_state_dict'])
    sae.eval()
    act_mean = np.array(cfg['act_mean'], dtype=np.float32)   # shape (1, d_in)
    logger.info('Loaded SAE from %s  d_in=%d  d_sae=%d', path, cfg['d_in'], cfg['d_sae'])
    return sae, act_mean


# ── W2V EMI ───────────────────────────────────────────────────────────────────
def compute_emi(text, wv, ev_centroid, in_centroid):
    """Word2Vec EMI score for a single speech text. Returns None if no vocab overlap."""
    words = text.lower().split()
    vecs  = [wv[w] for w in words if w in wv]
    if not vecs:
        return None
    speech_vec = np.mean(vecs, axis=0)
    def _cos(a, b):
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
    return _cos(speech_vec, ev_centroid) - _cos(speech_vec, in_centroid)


# ── SAE feature extraction ────────────────────────────────────────────────────
def compute_sae_features(sae, activations, act_mean, batch_size=SAE_BATCH_SIZE):
    """Pass activations through SAE encoder. Returns (N, d_sae) float32 array."""
    sae.eval()
    n        = len(activations)
    features = np.zeros((n, sae.d_sae), dtype=np.float32)
    acts_c   = activations.astype(np.float32) - act_mean   # subtract mean
    with torch.no_grad():
        for start in range(0, n, batch_size):
            end   = min(start + batch_size, n)
            batch = torch.from_numpy(acts_c[start:end]).to(device)
            f     = sae.encode(batch)
            features[start:end] = f.float().cpu().numpy()
    return features


# ── Cosine EMI in sparse feature space ───────────────────────────────────────
def compute_emi_cosine(speech_features, ev_prototype, in_prototype):
    """EMI = cos(speech_sparse, evidence_proto) - cos(speech_sparse, intuition_proto)."""
    ev_norm      = ev_prototype / np.linalg.norm(ev_prototype)
    in_norm      = in_prototype / np.linalg.norm(in_prototype)
    speech_norms = np.linalg.norm(speech_features, axis=1, keepdims=True)
    speech_norms = np.where(speech_norms == 0, 1.0, speech_norms)
    speech_unit  = speech_features / speech_norms
    return (speech_unit @ ev_norm - speech_unit @ in_norm).astype(np.float32)


# ── CSV speech retrieval ──────────────────────────────────────────────────────
def collect_texts_from_csv(data_file, target_csv_rows):
    """
    Single streaming pass through CSV to collect texts at the given row indices.
    target_csv_rows: list of 0-indexed row numbers (after header).
    Returns dict {csv_row: text}.
    """
    target_set = set(target_csv_rows)
    max_row    = max(target_csv_rows)
    collected  = {}
    with open(data_file, 'r', newline='', encoding='utf-8', errors='replace') as fh:
        reader = csv.DictReader(fh)
        for csv_row, row in enumerate(reader):
            if csv_row in target_set:
                collected[csv_row] = row.get('text', '') or ''
            if csv_row >= max_row:
                break
    return collected


# ── Log statistics helper ─────────────────────────────────────────────────────
def log_stats(arr, name):
    logger.info('%s  shape=%s  mean=%.4f  std=%.4f  min=%.4f  max=%.4f',
                name, arr.shape, arr.mean(), arr.std(), arr.min(), arr.max())


print('Helper functions and SparseAutoencoder defined.')

## Section 1 — Phase 0: Word2Vec EMI Scoring of Full Corpus

In [ ]:
# Cell 5 — Load W2V model
W2V_SCORES_PATH   = os.path.join(RESULTS_DIR, 'w2v_scores_full_corpus.npy')
W2V_METADATA_PATH = os.path.join(RESULTS_DIR, 'w2v_full_corpus_metadata.json')

if os.path.isfile(W2V_SCORES_PATH) and os.path.isfile(W2V_METADATA_PATH):
    print('W2V scores already exist — skipping W2V model load.')
    wv = None  # will not be needed
else:
    from gensim.models import KeyedVectors
    logger.info('Loading W2V model from %s  (1-2 min)...', W2V_PATH)
    t0 = time.time()
    wv = KeyedVectors.load_word2vec_format(W2V_PATH, binary=False, no_header=True)
    logger.info('W2V loaded: vocab=%d  dim=%d  (%.1fs)', len(wv), wv.vector_size, time.time()-t0)

In [ ]:
# Cell 6 — Build W2V centroids and score full corpus (skip-if-exists)

if os.path.isfile(W2V_SCORES_PATH) and os.path.isfile(W2V_METADATA_PATH):
    print('Loading W2V scores from cache...')
else:
    evidence_seeds = [
        'accurate', 'exact', 'intelligence', 'precise', 'search',
        'analyse', 'examination', 'investigate', 'procedure', 'show',
        'analysis', 'examine', 'investigation', 'process', 'statistics',
        'correct', 'expert', 'knowledge', 'proof', 'study',
        'correction', 'explore', 'lab', 'question', 'trial',
        'data', 'fact', 'learn', 'read', 'real',
        'dossier', 'find', 'logic', 'reason', 'true',
        'education', 'findings', 'logical', 'research', 'truth',
        'evidence', 'information', 'method', 'science', 'truthful',
        'evident', 'inquiry', 'pinpoint', 'scientific',
    ]
    intuition_seeds = [
        'advice', 'doubt', 'mislead', 'suggestion', 'belief',
        'fake', 'mistaken', 'suspicion', 'believe',
        'mistrust', 'view', 'bogus', 'feeling', 'opinion',
        'viewpoint', 'genuine', 'perspective', 'wrong',
        'deceive', 'guess', 'phony', 'deception', 'gut',
        'dishonest', 'instinct', 'propaganda', 'dishonesty',
        'intuition', 'sense', 'distrust', 'lie', 'suggest',
    ]

    missing_ev = [w for w in evidence_seeds  if w not in wv]
    missing_in = [w for w in intuition_seeds if w not in wv]
    logger.info('Evidence seeds: %d/%d in vocab', len(evidence_seeds)-len(missing_ev), len(evidence_seeds))
    logger.info('Intuition seeds: %d/%d in vocab', len(intuition_seeds)-len(missing_in), len(intuition_seeds))
    if missing_ev: logger.warning('Missing evidence seeds: %s', missing_ev)
    if missing_in: logger.warning('Missing intuition seeds: %s', missing_in)

    ev_centroid = np.array([wv[w] for w in evidence_seeds  if w in wv], dtype=np.float32).mean(axis=0)
    in_centroid = np.array([wv[w] for w in intuition_seeds if w in wv], dtype=np.float32).mean(axis=0)

    def _cos(a, b):
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
    logger.info('Centroid cosine similarity: %.4f', _cos(ev_centroid, in_centroid))

    logger.info('Streaming through %s to score all speeches...', DATA_CSV)
    t0      = time.time()
    scores   = []
    metadata = []
    n_skipped = 0

    with open(DATA_CSV, 'r', newline='', encoding='utf-8', errors='replace') as fh:
        reader = csv.DictReader(fh)
        for csv_row, row in enumerate(reader):
            text  = row.get('text', '') or ''
            yr_s  = row.get('year', '') or ''
            party = row.get('party', '') or ''
            try:
                year = int(float(yr_s))
            except (ValueError, TypeError):
                n_skipped += 1
                continue
            if not text or not (1870 <= year <= 2030):
                n_skipped += 1
                continue

            emi = compute_emi(text, wv, ev_centroid, in_centroid)
            if emi is None:
                n_skipped += 1
                continue

            scores.append(emi)
            metadata.append({
                'csv_row': csv_row,
                'year':    year,
                'party':   party,
                'decade':  (year // 10) * 10,
            })

            if len(scores) % 500_000 == 0:
                elapsed = time.time() - t0
                logger.info('  Scored %d speeches so far  (%.1f min)', len(scores), elapsed/60)

    scores_arr = np.array(scores, dtype=np.float32)
    np.save(W2V_SCORES_PATH, scores_arr)
    with open(W2V_METADATA_PATH, 'w') as fh:
        json.dump(metadata, fh)

    elapsed = time.time() - t0
    logger.info('W2V scoring done: %d scored, %d skipped  (%.1f min)',
                len(scores), n_skipped, elapsed/60)
    del wv
    gc.collect()

# always load from disk
w2v_scores   = np.load(W2V_SCORES_PATH)
w2v_metadata = json.load(open(W2V_METADATA_PATH))
assert len(w2v_scores) == len(w2v_metadata), 'Scores/metadata length mismatch'
assert not np.isnan(w2v_scores).any(), 'NaN in W2V scores'
assert not np.isinf(w2v_scores).any(), 'Inf in W2V scores'

log_stats(w2v_scores, 'W2V scores')
years  = [m['year']  for m in w2v_metadata]
parties = [m['party'] for m in w2v_metadata]
logger.info('Year range: %d – %d', min(years), max(years))
from collections import Counter
logger.info('Party distribution: %s', dict(Counter(parties).most_common(5)))
print(f'W2V scores loaded: {w2v_scores.shape}  ({len(w2v_scores):,} speeches)')

## Section 2 — Phase 1: Select Exemplars

In [ ]:
# Cell 7 — Select top 100K at each pole
EXEMPLAR_PATH = os.path.join(RESULTS_DIR, 'phase1_exemplar_indices.json')

if os.path.isfile(EXEMPLAR_PATH):
    exemplars = json.load(open(EXEMPLAR_PATH))
    print('Loaded exemplar indices from cache.')
else:
    sorted_idx        = np.argsort(w2v_scores)
    top_intuition_pos = sorted_idx[:PHASE1_TOP_N_PER_POLE].tolist()
    top_evidence_pos  = sorted_idx[-PHASE1_TOP_N_PER_POLE:].tolist()

    # retrieve csv_row for each position
    ev_csv_rows = [w2v_metadata[i]['csv_row'] for i in top_evidence_pos]
    in_csv_rows = [w2v_metadata[i]['csv_row'] for i in top_intuition_pos]

    exemplars = {
        'evidence_pos':        top_evidence_pos,
        'intuition_pos':       top_intuition_pos,
        'evidence_csv_rows':   ev_csv_rows,
        'intuition_csv_rows':  in_csv_rows,
        'w2v_evidence_range':  [float(w2v_scores[top_evidence_pos].min()),
                                 float(w2v_scores[top_evidence_pos].max())],
        'w2v_intuition_range': [float(w2v_scores[top_intuition_pos].min()),
                                 float(w2v_scores[top_intuition_pos].max())],
    }
    with open(EXEMPLAR_PATH, 'w') as fh:
        json.dump(exemplars, fh)
    print('Saved exemplar indices.')

    # distribution check
    for pole, pos in [('Evidence', exemplars['evidence_pos']),
                      ('Intuition', exemplars['intuition_pos'])]:
        decades = [(w2v_metadata[i]['decade'] // 10) * 10 for i in pos]
        dc = Counter(decades)
        if len(dc) < 5:
            logger.warning('%s exemplars heavily skewed by decade: %s', pole, dict(dc.most_common(5)))
        pts = [w2v_metadata[i]['party'] for i in pos]
        pc = Counter(pts)
        if pc.most_common(1)[0][1] > 0.9 * len(pos):
            logger.warning('%s exemplars heavily skewed by party: %s', pole, dict(pc.most_common(3)))

print(f"Evidence W2V range:  {exemplars['w2v_evidence_range']}")
print(f"Intuition W2V range: {exemplars['w2v_intuition_range']}")
print(f"Evidence exemplars:  {len(exemplars['evidence_csv_rows']):,}")
print(f"Intuition exemplars: {len(exemplars['intuition_csv_rows']):,}")

## Section 3 — Phase 1: Collect Activations for Exemplars

In [ ]:
# Cell 8 — Define collect_activations_for_csv_rows

def collect_activations_for_csv_rows(model, tokenizer, ordered_csv_rows,
                                      data_file, max_length, batch_size, hidden_dim):
    """
    Stream through data_file once (sorted pass), collecting texts at ordered_csv_rows.
    ordered_csv_rows: list of csv_row indices IN THE ORDER they should appear in output.
    Returns numpy array shape (len(ordered_csv_rows), hidden_dim).
    """
    model.eval()
    model = model.to(device)

    # Build a mapping from csv_row → list of output positions
    csv_row_to_positions = defaultdict(list)
    for out_pos, csv_row in enumerate(ordered_csv_rows):
        csv_row_to_positions[csv_row].append(out_pos)

    target_set = set(ordered_csv_rows)
    max_row    = max(ordered_csv_rows)

    logger.info('Streaming CSV to collect %d texts (max csv_row=%d)...', len(target_set), max_row)

    # Single pass: collect texts in the order they appear in CSV
    row_text_pairs = []   # (csv_row, text)
    with open(data_file, 'r', newline='', encoding='utf-8', errors='replace') as fh:
        reader = csv.DictReader(fh)
        for csv_row, row in enumerate(reader):
            if csv_row in target_set:
                row_text_pairs.append((csv_row, row.get('text', '') or ''))
            if csv_row >= max_row:
                break

    logger.info('Collected %d texts from CSV (expected %d)', len(row_text_pairs), len(target_set))

    n_total = len(ordered_csv_rows)
    output  = np.zeros((n_total, hidden_dim), dtype=np.float32)

    # Build a lookup: csv_row → text
    text_by_csvrow = {csv_row: text for csv_row, text in row_text_pairs}

    # Batch inference in the order of ordered_csv_rows
    t0 = time.time()
    for batch_start in range(0, n_total, batch_size):
        batch_end  = min(batch_start + batch_size, n_total)
        batch_rows = ordered_csv_rows[batch_start:batch_end]
        texts      = [text_by_csvrow.get(r, '') for r in batch_rows]

        inputs = tokenizer(
            texts,
            return_tensors='pt',
            truncation=True,
            max_length=max_length,
            padding=True,
        ).to(device)

        with torch.no_grad():
            out = model(**inputs, output_hidden_states=True)

        last_hidden = out.hidden_states[-1]
        mask        = inputs['attention_mask'].unsqueeze(-1).float()
        pooled      = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1)
        output[batch_start:batch_end] = pooled.float().cpu().numpy()

        del out, last_hidden, pooled
        torch.cuda.empty_cache()

        if batch_start % (batch_size * 100) == 0:
            elapsed = time.time() - t0
            rate    = batch_end / elapsed if elapsed > 0 else 0
            logger.info('  %d/%d  (%.0f speeches/sec)', batch_end, n_total, rate)

    return output

print('collect_activations_for_csv_rows defined.')

In [ ]:
# Cell 9 — Collect BERT Phase 1 activations (skip-if-exists)
BERT_P1_ACT_PATH = os.path.join(RESULTS_DIR, 'bert_phase1_activations.npy')

if os.path.isfile(BERT_P1_ACT_PATH):
    bert_phase1_acts = np.load(BERT_P1_ACT_PATH)
    print(f'Loaded cached BERT Phase 1 activations: {bert_phase1_acts.shape}')
else:
    from transformers import BertModel, BertTokenizer

    logger.info('Loading BERT from %s...', BERT_MODEL_DIR)
    bert_tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_DIR)
    bert_model     = BertModel.from_pretrained(BERT_MODEL_DIR, output_hidden_states=True)
    bert_model.eval()

    # evidence first, then intuition (order must match prototype construction)
    ordered_csv_rows = exemplars['evidence_csv_rows'] + exemplars['intuition_csv_rows']

    bert_phase1_acts = collect_activations_for_csv_rows(
        model          = bert_model,
        tokenizer      = bert_tokenizer,
        ordered_csv_rows = ordered_csv_rows,
        data_file      = DATA_CSV,
        max_length     = 256,
        batch_size     = BERT_BATCH_SIZE,
        hidden_dim     = 768,
    )

    assert bert_phase1_acts.shape == (len(ordered_csv_rows), 768), \
        f'Unexpected shape: {bert_phase1_acts.shape}'
    assert not np.isnan(bert_phase1_acts).any(), 'NaN in BERT Phase 1 activations'

    np.save(BERT_P1_ACT_PATH, bert_phase1_acts)
    logger.info('Saved BERT Phase 1 activations: %s', bert_phase1_acts.shape)

    del bert_model, bert_tokenizer
    torch.cuda.empty_cache()
    gc.collect()

log_stats(bert_phase1_acts, 'BERT Phase 1 activations')
print(f'BERT Phase 1 activations: {bert_phase1_acts.shape}')

In [ ]:
# Cell 10 — Collect GPT-2 Phase 1 activations (skip-if-exists)
GPT2_P1_ACT_PATH = os.path.join(RESULTS_DIR, 'gpt2_phase1_activations.npy')

if os.path.isfile(GPT2_P1_ACT_PATH):
    gpt2_phase1_acts = np.load(GPT2_P1_ACT_PATH)
    print(f'Loaded cached GPT-2 Phase 1 activations: {gpt2_phase1_acts.shape}')
else:
    from transformers import GPT2Model, GPT2Tokenizer

    logger.info('Loading GPT-2 from %s...', GPT2_MODEL_DIR)
    gpt2_tokenizer = GPT2Tokenizer.from_pretrained(GPT2_MODEL_DIR)
    gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
    gpt2_model = GPT2Model.from_pretrained(GPT2_MODEL_DIR, output_hidden_states=True)
    gpt2_model.eval()

    ordered_csv_rows = exemplars['evidence_csv_rows'] + exemplars['intuition_csv_rows']

    gpt2_phase1_acts = collect_activations_for_csv_rows(
        model          = gpt2_model,
        tokenizer      = gpt2_tokenizer,
        ordered_csv_rows = ordered_csv_rows,
        data_file      = DATA_CSV,
        max_length     = 512,
        batch_size     = GPT2_BATCH_SIZE,
        hidden_dim     = 1024,
    )

    assert gpt2_phase1_acts.shape == (len(ordered_csv_rows), 1024), \
        f'Unexpected shape: {gpt2_phase1_acts.shape}'
    assert not np.isnan(gpt2_phase1_acts).any(), 'NaN in GPT-2 Phase 1 activations'

    np.save(GPT2_P1_ACT_PATH, gpt2_phase1_acts)
    logger.info('Saved GPT-2 Phase 1 activations: %s', gpt2_phase1_acts.shape)

    del gpt2_model, gpt2_tokenizer
    torch.cuda.empty_cache()
    gc.collect()

log_stats(gpt2_phase1_acts, 'GPT-2 Phase 1 activations')
print(f'GPT-2 Phase 1 activations: {gpt2_phase1_acts.shape}')

## Section 4 — Phase 1: Build Sparse Prototype Vectors

In [ ]:
# Cell 11 — BERT SAE features + prototypes
BERT_P1_FEAT_PATH = os.path.join(RESULTS_DIR, 'bert_phase1_sae_features.npy')
BERT_EV_PROTO_PATH = os.path.join(RESULTS_DIR, 'bert_evidence_prototype.npy')
BERT_IN_PROTO_PATH = os.path.join(RESULTS_DIR, 'bert_intuition_prototype.npy')

if (os.path.isfile(BERT_EV_PROTO_PATH)
        and os.path.isfile(BERT_IN_PROTO_PATH)
        and os.path.isfile(BERT_P1_FEAT_PATH)):
    bert_evidence_prototype  = np.load(BERT_EV_PROTO_PATH)
    bert_intuition_prototype = np.load(BERT_IN_PROTO_PATH)
    bert_phase1_features     = np.load(BERT_P1_FEAT_PATH)
    print('Loaded cached BERT prototypes and Phase 1 features.')
else:
    bert_sae, bert_act_mean = load_sae_checkpoint(SAE_BERT_PATH)

    bert_phase1_features = compute_sae_features(
        bert_sae, bert_phase1_acts, bert_act_mean
    )
    np.save(BERT_P1_FEAT_PATH, bert_phase1_features)

    active_frac = (bert_phase1_features > 0).mean()
    logger.info('BERT feature sparsity: %.2f%% active (target 1-5%%)', active_frac * 100)
    if active_frac > 0.1:
        logger.warning('BERT sparsity high (%.1f%%) — check SAE L1 coefficient', active_frac * 100)

    n_ev = PHASE1_TOP_N_PER_POLE   # evidence rows come first
    bert_evidence_prototype  = bert_phase1_features[:n_ev].mean(axis=0)
    bert_intuition_prototype = bert_phase1_features[n_ev:].mean(axis=0)

    np.save(BERT_EV_PROTO_PATH, bert_evidence_prototype)
    np.save(BERT_IN_PROTO_PATH, bert_intuition_prototype)

    del bert_sae
    torch.cuda.empty_cache()

ev_act = (bert_evidence_prototype  > 0).sum()
in_act = (bert_intuition_prototype > 0).sum()
ev_n   = bert_evidence_prototype.norm() if hasattr(bert_evidence_prototype, 'norm') \
         else float(np.linalg.norm(bert_evidence_prototype))
proto_cos = float((bert_evidence_prototype / np.linalg.norm(bert_evidence_prototype))
                  @ (bert_intuition_prototype / np.linalg.norm(bert_intuition_prototype)))
logger.info('BERT evidence prototype:  %d active / %d total features', ev_act, len(bert_evidence_prototype))
logger.info('BERT intuition prototype: %d active / %d total features', in_act, len(bert_intuition_prototype))
logger.info('BERT prototype cosine similarity: %.4f (lower = better)', proto_cos)

# free Phase 1 activations — no longer needed
del bert_phase1_acts, bert_phase1_features
gc.collect()

print(f'BERT evidence prototype:  {ev_act} active features')
print(f'BERT intuition prototype: {in_act} active features')
print(f'BERT prototype cosine:    {proto_cos:.4f}')

In [ ]:
# Cell 12 — GPT-2 SAE features + prototypes
GPT2_P1_FEAT_PATH  = os.path.join(RESULTS_DIR, 'gpt2_phase1_sae_features.npy')
GPT2_EV_PROTO_PATH = os.path.join(RESULTS_DIR, 'gpt2_evidence_prototype.npy')
GPT2_IN_PROTO_PATH = os.path.join(RESULTS_DIR, 'gpt2_intuition_prototype.npy')

if (os.path.isfile(GPT2_EV_PROTO_PATH)
        and os.path.isfile(GPT2_IN_PROTO_PATH)
        and os.path.isfile(GPT2_P1_FEAT_PATH)):
    gpt2_evidence_prototype  = np.load(GPT2_EV_PROTO_PATH)
    gpt2_intuition_prototype = np.load(GPT2_IN_PROTO_PATH)
    gpt2_phase1_features     = np.load(GPT2_P1_FEAT_PATH)
    print('Loaded cached GPT-2 prototypes and Phase 1 features.')
else:
    gpt2_sae, gpt2_act_mean = load_sae_checkpoint(SAE_GPT2_PATH)

    gpt2_phase1_features = compute_sae_features(
        gpt2_sae, gpt2_phase1_acts, gpt2_act_mean
    )
    np.save(GPT2_P1_FEAT_PATH, gpt2_phase1_features)

    active_frac = (gpt2_phase1_features > 0).mean()
    logger.info('GPT-2 feature sparsity: %.2f%% active (target 1-5%%)', active_frac * 100)
    if active_frac > 0.1:
        logger.warning('GPT-2 sparsity high (%.1f%%) — check SAE L1 coefficient', active_frac * 100)

    n_ev = PHASE1_TOP_N_PER_POLE
    gpt2_evidence_prototype  = gpt2_phase1_features[:n_ev].mean(axis=0)
    gpt2_intuition_prototype = gpt2_phase1_features[n_ev:].mean(axis=0)

    np.save(GPT2_EV_PROTO_PATH, gpt2_evidence_prototype)
    np.save(GPT2_IN_PROTO_PATH, gpt2_intuition_prototype)

    del gpt2_sae
    torch.cuda.empty_cache()

ev_act    = (gpt2_evidence_prototype  > 0).sum()
in_act    = (gpt2_intuition_prototype > 0).sum()
proto_cos = float((gpt2_evidence_prototype / np.linalg.norm(gpt2_evidence_prototype))
                  @ (gpt2_intuition_prototype / np.linalg.norm(gpt2_intuition_prototype)))
logger.info('GPT-2 evidence prototype:  %d active / %d total features', ev_act, len(gpt2_evidence_prototype))
logger.info('GPT-2 intuition prototype: %d active / %d total features', in_act, len(gpt2_intuition_prototype))
logger.info('GPT-2 prototype cosine similarity: %.4f (lower = better)', proto_cos)

del gpt2_phase1_acts, gpt2_phase1_features
gc.collect()

print(f'GPT-2 evidence prototype:  {ev_act} active features')
print(f'GPT-2 intuition prototype: {in_act} active features')
print(f'GPT-2 prototype cosine:    {proto_cos:.4f}')

## Section 5 — Phase 2: Stratified Sampling

In [ ]:
# Cell 13 — Build decade-stratified Phase 2 sample
PHASE2_INDICES_PATH = os.path.join(RESULTS_DIR, 'phase2_stratified_indices.json')

if os.path.isfile(PHASE2_INDICES_PATH):
    phase2_data = json.load(open(PHASE2_INDICES_PATH))
    print('Loaded Phase 2 indices from cache.')
else:
    # group scored-speech positions by decade
    by_decade = defaultdict(list)
    for idx, meta in enumerate(w2v_metadata):
        decade = meta.get('decade')
        if decade is not None and PHASE2_DECADE_START <= decade <= PHASE2_DECADE_END:
            by_decade[int(decade)].append(idx)

    rng     = np.random.default_rng(SEED)
    pos_list      = []   # position in w2v_metadata/w2v_scores
    csv_row_list  = []   # corresponding csv_row in filtered_speeches.csv

    for decade in sorted(by_decade):
        available = by_decade[decade]
        n_take    = min(PHASE2_SPEECHES_PER_DECADE, len(available))
        chosen    = rng.choice(available, size=n_take, replace=False).tolist()
        pos_list.extend(chosen)
        csv_row_list.extend([w2v_metadata[i]['csv_row'] for i in chosen])
        logger.info('Decade %d: %d/%d sampled', decade, n_take, len(available))

    phase2_data = {'positions': pos_list, 'csv_rows': csv_row_list}
    with open(PHASE2_INDICES_PATH, 'w') as fh:
        json.dump(phase2_data, fh)

phase2_positions = phase2_data['positions']
phase2_csv_rows  = phase2_data['csv_rows']
print(f'Phase 2 stratified sample: {len(phase2_positions):,} speeches')

## Section 6 — Phase 2: Collect Activations and Score

In [ ]:
# Cell 14 — BERT Phase 2 activations (skip-if-exists)
BERT_P2_ACT_PATH = os.path.join(RESULTS_DIR, 'bert_phase2_activations.npy')

if os.path.isfile(BERT_P2_ACT_PATH):
    bert_phase2_acts = np.load(BERT_P2_ACT_PATH)
    print(f'Loaded cached BERT Phase 2 activations: {bert_phase2_acts.shape}')
else:
    from transformers import BertModel, BertTokenizer

    logger.info('Loading BERT from %s...', BERT_MODEL_DIR)
    bert_tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_DIR)
    bert_model     = BertModel.from_pretrained(BERT_MODEL_DIR, output_hidden_states=True)
    bert_model.eval()

    bert_phase2_acts = collect_activations_for_csv_rows(
        model          = bert_model,
        tokenizer      = bert_tokenizer,
        ordered_csv_rows = phase2_csv_rows,
        data_file      = DATA_CSV,
        max_length     = 256,
        batch_size     = BERT_BATCH_SIZE,
        hidden_dim     = 768,
    )

    assert not np.isnan(bert_phase2_acts).any(), 'NaN in BERT Phase 2 activations'
    np.save(BERT_P2_ACT_PATH, bert_phase2_acts)
    logger.info('Saved BERT Phase 2 activations: %s', bert_phase2_acts.shape)

    del bert_model, bert_tokenizer
    torch.cuda.empty_cache()
    gc.collect()

print(f'BERT Phase 2 activations: {bert_phase2_acts.shape}')

In [ ]:
# Cell 15 — BERT Phase 2 SAE features + EMI scores
BERT_P2_FEAT_PATH = os.path.join(RESULTS_DIR, 'bert_phase2_sae_features.npy')
BERT_P2_EMI_PATH  = os.path.join(RESULTS_DIR, 'bert_phase2_emi_scores.npy')

if os.path.isfile(BERT_P2_EMI_PATH):
    bert_phase2_emi = np.load(BERT_P2_EMI_PATH)
    print(f'Loaded cached BERT Phase 2 EMI scores: {bert_phase2_emi.shape}')
else:
    bert_sae, bert_act_mean = load_sae_checkpoint(SAE_BERT_PATH)

    if os.path.isfile(BERT_P2_FEAT_PATH):
        bert_phase2_features = np.load(BERT_P2_FEAT_PATH)
    else:
        bert_phase2_features = compute_sae_features(
            bert_sae, bert_phase2_acts, bert_act_mean
        )
        np.save(BERT_P2_FEAT_PATH, bert_phase2_features)

    active_frac = (bert_phase2_features > 0).mean()
    logger.info('BERT Phase 2 sparsity: %.2f%% active', active_frac * 100)

    bert_phase2_emi = compute_emi_cosine(
        bert_phase2_features,
        bert_evidence_prototype,
        bert_intuition_prototype,
    )
    np.save(BERT_P2_EMI_PATH, bert_phase2_emi)

    del bert_sae, bert_phase2_features
    torch.cuda.empty_cache()

phase2_w2v_scores = w2v_scores[phase2_positions]
r, p = pearsonr(phase2_w2v_scores, bert_phase2_emi)
if r < 0:
    logger.warning('BERT Phase 2 EMI has NEGATIVE correlation with W2V (r=%.4f) — check prototype order', r)
else:
    logger.info('BERT vs W2V Pearson r=%.4f  p=%.2e', r, p)

del bert_phase2_acts
gc.collect()

log_stats(bert_phase2_emi, 'BERT Phase 2 EMI')
print(f'BERT Phase 2 EMI: {bert_phase2_emi.shape}  Pearson r={r:.4f}')

In [ ]:
# Cell 16 — GPT-2 Phase 2 activations (skip-if-exists)
GPT2_P2_ACT_PATH = os.path.join(RESULTS_DIR, 'gpt2_phase2_activations.npy')

if os.path.isfile(GPT2_P2_ACT_PATH):
    gpt2_phase2_acts = np.load(GPT2_P2_ACT_PATH)
    print(f'Loaded cached GPT-2 Phase 2 activations: {gpt2_phase2_acts.shape}')
else:
    from transformers import GPT2Model, GPT2Tokenizer

    logger.info('Loading GPT-2 from %s...', GPT2_MODEL_DIR)
    gpt2_tokenizer = GPT2Tokenizer.from_pretrained(GPT2_MODEL_DIR)
    gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
    gpt2_model = GPT2Model.from_pretrained(GPT2_MODEL_DIR, output_hidden_states=True)
    gpt2_model.eval()

    gpt2_phase2_acts = collect_activations_for_csv_rows(
        model          = gpt2_model,
        tokenizer      = gpt2_tokenizer,
        ordered_csv_rows = phase2_csv_rows,
        data_file      = DATA_CSV,
        max_length     = 512,
        batch_size     = GPT2_BATCH_SIZE,
        hidden_dim     = 1024,
    )

    assert not np.isnan(gpt2_phase2_acts).any(), 'NaN in GPT-2 Phase 2 activations'
    np.save(GPT2_P2_ACT_PATH, gpt2_phase2_acts)
    logger.info('Saved GPT-2 Phase 2 activations: %s', gpt2_phase2_acts.shape)

    del gpt2_model, gpt2_tokenizer
    torch.cuda.empty_cache()
    gc.collect()

print(f'GPT-2 Phase 2 activations: {gpt2_phase2_acts.shape}')

In [ ]:
# Cell 17 — GPT-2 Phase 2 SAE features + EMI scores
GPT2_P2_FEAT_PATH = os.path.join(RESULTS_DIR, 'gpt2_phase2_sae_features.npy')
GPT2_P2_EMI_PATH  = os.path.join(RESULTS_DIR, 'gpt2_phase2_emi_scores.npy')

if os.path.isfile(GPT2_P2_EMI_PATH):
    gpt2_phase2_emi = np.load(GPT2_P2_EMI_PATH)
    print(f'Loaded cached GPT-2 Phase 2 EMI scores: {gpt2_phase2_emi.shape}')
else:
    gpt2_sae, gpt2_act_mean = load_sae_checkpoint(SAE_GPT2_PATH)

    if os.path.isfile(GPT2_P2_FEAT_PATH):
        gpt2_phase2_features = np.load(GPT2_P2_FEAT_PATH)
    else:
        gpt2_phase2_features = compute_sae_features(
            gpt2_sae, gpt2_phase2_acts, gpt2_act_mean
        )
        np.save(GPT2_P2_FEAT_PATH, gpt2_phase2_features)

    active_frac = (gpt2_phase2_features > 0).mean()
    logger.info('GPT-2 Phase 2 sparsity: %.2f%% active', active_frac * 100)

    gpt2_phase2_emi = compute_emi_cosine(
        gpt2_phase2_features,
        gpt2_evidence_prototype,
        gpt2_intuition_prototype,
    )
    np.save(GPT2_P2_EMI_PATH, gpt2_phase2_emi)

    del gpt2_sae, gpt2_phase2_features
    torch.cuda.empty_cache()

r, p = pearsonr(phase2_w2v_scores, gpt2_phase2_emi)
if r < 0:
    logger.warning('GPT-2 Phase 2 EMI has NEGATIVE correlation with W2V (r=%.4f)', r)
else:
    logger.info('GPT-2 vs W2V Pearson r=%.4f  p=%.2e', r, p)

del gpt2_phase2_acts
gc.collect()

log_stats(gpt2_phase2_emi, 'GPT-2 Phase 2 EMI')
print(f'GPT-2 Phase 2 EMI: {gpt2_phase2_emi.shape}  Pearson r={r:.4f}')

## Section 7 — Top Feature Analysis

In [ ]:
# Cell 18 — Identify top discriminative SAE features per model
TOP_FEATURES_PATH = os.path.join(RESULTS_DIR, 'prototype_top_features.json')

def top_features_for_model(ev_proto, in_proto, model_name, top_k=20):
    differential = ev_proto - in_proto

    sorted_desc = np.argsort(differential)[::-1]
    sorted_asc  = np.argsort(differential)

    ev_features = [
        {
            'feature_id':             int(idx),
            'differential':           float(differential[idx]),
            'evidence_activation':    float(ev_proto[idx]),
            'intuition_activation':   float(in_proto[idx]),
        }
        for idx in sorted_desc[:top_k]
    ]
    in_features = [
        {
            'feature_id':             int(idx),
            'differential':           float(differential[idx]),
            'evidence_activation':    float(ev_proto[idx]),
            'intuition_activation':   float(in_proto[idx]),
        }
        for idx in sorted_asc[:top_k]
    ]
    return {'top_evidence_features': ev_features, 'top_intuition_features': in_features}


if os.path.isfile(TOP_FEATURES_PATH):
    top_features = json.load(open(TOP_FEATURES_PATH))
    print('Loaded cached top features.')
else:
    top_features = {
        'bert': top_features_for_model(bert_evidence_prototype, bert_intuition_prototype, 'bert'),
        'gpt2': top_features_for_model(gpt2_evidence_prototype, gpt2_intuition_prototype, 'gpt2'),
    }
    with open(TOP_FEATURES_PATH, 'w') as fh:
        json.dump(top_features, fh, indent=2)

print('BERT top-3 evidence features:')
for f in top_features['bert']['top_evidence_features'][:3]:
    print(f"  feature {f['feature_id']:5d}  diff={f['differential']:+.4f}  "
          f"ev={f['evidence_activation']:.4f}  in={f['intuition_activation']:.4f}")
print('BERT top-3 intuition features:')
for f in top_features['bert']['top_intuition_features'][:3]:
    print(f"  feature {f['feature_id']:5d}  diff={f['differential']:+.4f}  "
          f"ev={f['evidence_activation']:.4f}  in={f['intuition_activation']:.4f}")
print()
print('GPT-2 top-3 evidence features:')
for f in top_features['gpt2']['top_evidence_features'][:3]:
    print(f"  feature {f['feature_id']:5d}  diff={f['differential']:+.4f}  "
          f"ev={f['evidence_activation']:.4f}  in={f['intuition_activation']:.4f}")

## Section 8 — Cross-Method Validation

In [ ]:
# Cell 19 — Build comparison CSV and compute validation metrics
COMPARISON_CSV_PATH = os.path.join(RESULTS_DIR, 'emi_comparison_phase2.csv')
VALIDATION_PATH     = os.path.join(RESULTS_DIR, 'validation_metrics.json')

phase2_metadata = [w2v_metadata[i] for i in phase2_positions]

df = pd.DataFrame({
    'speech_pos': phase2_positions,
    'csv_row':    phase2_csv_rows,
    'year':       [m.get('year')   for m in phase2_metadata],
    'party':      [m.get('party')  for m in phase2_metadata],
    'decade':     [m.get('decade') for m in phase2_metadata],
    'w2v_emi':    phase2_w2v_scores,
    'bert_emi':   bert_phase2_emi,
    'gpt2_emi':   gpt2_phase2_emi,
})

df.to_csv(COMPARISON_CSV_PATH, index=False)
print(f'Saved comparison CSV: {df.shape}')
print(df[['w2v_emi', 'bert_emi', 'gpt2_emi']].describe().round(4))

# Cross-method correlations
correlations = {}
for name, (a, b) in [
    ('w2v_vs_bert',  (df.w2v_emi,  df.bert_emi)),
    ('w2v_vs_gpt2',  (df.w2v_emi,  df.gpt2_emi)),
    ('bert_vs_gpt2', (df.bert_emi, df.gpt2_emi)),
]:
    try:
        pr, pp = pearsonr(a, b)
        sr, sp = spearmanr(a, b)
        correlations[name] = {
            'pearson_r':    float(pr), 'pearson_p':   float(pp),
            'spearman_rho': float(sr), 'spearman_p':  float(sp),
        }
        logger.info('%s  Pearson r=%.4f  Spearman rho=%.4f', name, pr, sr)
    except Exception as exc:
        logger.warning('Correlation failed for %s: %s', name, exc)
        correlations[name] = {'error': str(exc)}

# AUC: do contextual EMI scores identify top quartile of W2V EMI?
threshold = float(np.percentile(df.w2v_emi, 75))
binary    = (df.w2v_emi > threshold).astype(int)
aucs = {}
for model_name, scores in [('bert_vs_w2v', df.bert_emi), ('gpt2_vs_w2v', df.gpt2_emi)]:
    try:
        aucs[model_name] = float(roc_auc_score(binary, scores))
    except Exception as exc:
        logger.warning('AUC failed for %s: %s', model_name, exc)
        aucs[model_name] = None

bert_proto_cos  = float((bert_evidence_prototype / np.linalg.norm(bert_evidence_prototype))
                        @ (bert_intuition_prototype / np.linalg.norm(bert_intuition_prototype)))
gpt2_proto_cos  = float((gpt2_evidence_prototype / np.linalg.norm(gpt2_evidence_prototype))
                        @ (gpt2_intuition_prototype / np.linalg.norm(gpt2_intuition_prototype)))

validation_metrics = {
    'correlations': correlations,
    'auc_top_quartile': aucs,
    'prototype_cosine_similarity': {
        'bert': bert_proto_cos,
        'gpt2': gpt2_proto_cos,
    },
    'exemplar_selection': {
        'phase1_top_n_per_pole':        PHASE1_TOP_N_PER_POLE,
        'w2v_evidence_range':           exemplars['w2v_evidence_range'],
        'w2v_intuition_range':          exemplars['w2v_intuition_range'],
    },
    'phase2': {
        'speeches_per_decade':          PHASE2_SPEECHES_PER_DECADE,
        'decade_range':                 [PHASE2_DECADE_START, PHASE2_DECADE_END],
        'total_speeches':               len(df),
    },
}

with open(VALIDATION_PATH, 'w') as fh:
    json.dump(validation_metrics, fh, indent=2)

print('\nValidation metrics:')
print(json.dumps(validation_metrics, indent=2))

In [ ]:
# Cell 20 — Final summary
print('=' * 60)
print('SPARSE SAE EMI PIPELINE — COMPLETE')
print('=' * 60)

expected_files = [
    ('w2v_scores_full_corpus.npy',       'W2V scores (full corpus)'),
    ('w2v_full_corpus_metadata.json',    'W2V metadata with csv_row'),
    ('phase1_exemplar_indices.json',     'Phase 1 exemplar csv_rows'),
    ('bert_phase1_activations.npy',      'BERT Phase 1 activations'),
    ('gpt2_phase1_activations.npy',      'GPT-2 Phase 1 activations'),
    ('bert_phase1_sae_features.npy',     'BERT Phase 1 SAE features'),
    ('gpt2_phase1_sae_features.npy',     'GPT-2 Phase 1 SAE features'),
    ('bert_evidence_prototype.npy',      'BERT evidence prototype'),
    ('bert_intuition_prototype.npy',     'BERT intuition prototype'),
    ('gpt2_evidence_prototype.npy',      'GPT-2 evidence prototype'),
    ('gpt2_intuition_prototype.npy',     'GPT-2 intuition prototype'),
    ('phase2_stratified_indices.json',   'Phase 2 stratified indices'),
    ('bert_phase2_activations.npy',      'BERT Phase 2 activations'),
    ('gpt2_phase2_activations.npy',      'GPT-2 Phase 2 activations'),
    ('bert_phase2_sae_features.npy',     'BERT Phase 2 SAE features'),
    ('gpt2_phase2_sae_features.npy',     'GPT-2 Phase 2 SAE features'),
    ('bert_phase2_emi_scores.npy',       'BERT Phase 2 EMI scores'),
    ('gpt2_phase2_emi_scores.npy',       'GPT-2 Phase 2 EMI scores'),
    ('prototype_top_features.json',      'Top discriminative features'),
    ('emi_comparison_phase2.csv',        'Phase 2 comparison CSV'),
    ('validation_metrics.json',          'Validation metrics'),
]

all_ok = True
for fname, desc in expected_files:
    path = os.path.join(RESULTS_DIR, fname)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'  OK  {desc:<40} ({size_mb:.1f} MB)')
    else:
        print(f'  MISSING  {desc:<40} {path}')
        all_ok = False

print()
print('Key metrics:')
vm = json.load(open(VALIDATION_PATH))
for pair, vals in vm['correlations'].items():
    if 'pearson_r' in vals:
        print(f'  {pair:<20}  Pearson r={vals["pearson_r"]:+.4f}  '
              f'Spearman rho={vals["spearman_rho"]:+.4f}')
for model, auc in vm['auc_top_quartile'].items():
    if auc is not None:
        print(f'  AUC {model:<18}  {auc:.4f}')

print()
print('BERT prototype cosine:  ', vm['prototype_cosine_similarity']['bert'])
print('GPT-2 prototype cosine: ', vm['prototype_cosine_similarity']['gpt2'])
print()
if all_ok:
    print('All expected output files present.')
else:
    print('WARNING: Some output files are missing — rerun incomplete cells.')